# XGBoost injury classification

Fit on 80% of the prepared training split. Use the remaining 20% to select the multiclass softmax temperature by log loss and two ordinal risk cutoffs by macro F1. Evaluate on the shared test split.

- **Threshold metrics.** Apply the cutoffs to the expected class index, from 0 (Healthy) to 2 (Injured). Report overall accuracy and one-vs-rest accuracy, precision, recall (TPR), F1, and FPR.
- **Probability metrics and plots.** Report raw and calibrated Brier scores, calibrated one-vs-rest ROC AUC and trapezoidal PR AUC, and plot ROC, calibration, and precision-recall curves.
- **Combined score.** Give equal weight to macro ROC AUC and the normalized calibration score, calculated as 1 minus the calibrated multiclass Brier loss divided by 2.

A completed evaluation is saved to the local experiment tracker.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

working_dir = Path.cwd()
project_dir = next(
    candidate
    for candidate in (working_dir, working_dir.parent, working_dir / "Project")
    if (candidate / "experiments.py").is_file()
)
sys.path.insert(0, str(project_dir))
from experiments import build_model, run_test


In [ ]:
experiment_name = "xgboost"
model = build_model(experiment_name)
run_id, result = run_test(experiment_name, model, notes="", plot=True)

calibration = result["calibration"]
lower, upper = calibration["cutoffs"]
print(f"Validation temperature: {calibration['temperature']:.4f}")
print(f"Validation risk cutoffs: Low Risk {lower:.4f}, Injured {upper:.4f}")
print(f"Thresholded test accuracy: {result['accuracy']:.4f}")
display(pd.DataFrame.from_dict(result["threshold_metrics"], orient="index").round(4))
display(pd.DataFrame.from_dict(result["confusion_matrix"], orient="index"))
display(pd.DataFrame.from_dict(result["probability_metrics"], orient="index").round(4))
print(f"Raw multiclass Brier: {result['raw_brier']:.4f}")
print(f"Calibrated multiclass Brier: {result['calibrated_brier']:.4f}")
print(f"Calibrated multiclass log loss: {result['log_loss']:.4f}")
print(f"Weighted Brier?ROC AUC score: {result['combined_score']:.4f}")
print(f"Saved test run {run_id}")
plt.show()
